In [0]:
# Cell 1 — Parse deterministic_context into per-event entity arrays
# ============================================================================
# SUPPLEMENTARY TABLE  (STEP 1 of 2) — per-event enrichment
#
# Goal: a NEW per-event table built on top of tw_deviation_data_formatted_rdq.
# The source is at DATE/ROW grain (many rows per Event_Number); this pipeline
# collapses it to ONE ROW PER EVENT (pr_id = Event_Number). This cell assembles
# the per-event enrichment; later cells add LOC, topics, and root cause, then write.
#
# Enrichment pulled from deviation_embeddings (one row per pr_id):
#   - deterministic entities  → parsed out of deterministic_context into typed
#                               arrays (clinical_ids, documents/SOPs, acronyms,
#                               cros, devices, vendors)
#   - deterministic_context   → raw resolved-references blob (kept for reference)
#
# Sources:
#   - us_gmsgq_dev.gms_us_alyt.deviation_embeddings                (enrichment + text)
#   - us_gmsgq_dev.gms_us_mart.tw_deviation_data_formatted_rdq     (source rows)
#   - us_gmsgq_dev.gms_us_alyt.deviation_topic_assignments         (zero-shot BERTopic topics)
#   - us_gmsgq_dev.gms_us_alyt.deviation_root_cause                (LLM root-cause)
# ============================================================================
import re
import pandas as pd
from pyspark.sql import functions as F, types as T

CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"
MART    = "gms_us_mart"

EMB_TABLE    = f"{CATALOG}.{ALYT}.deviation_embeddings"
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"
OUTPUT_TABLE = f"{CATALOG}.{ALYT}.deviation_supplementary"

# Column on SOURCE_TABLE that identifies the event (cast to string == pr_id).
SRC_ID_COL   = "Event_Number"

# Topic assignments (pr_id, primary/secondary/tertiary_topic) from zero-shot
# BERTopic BGE-M3 model in topic_modeling.ipynb (Cell 7 writes this table).
TOPIC_TABLE  = f"{CATALOG}.{ALYT}.deviation_topic_assignments"

# Root-cause suggestions (pr_id, root_cause_category, ...) from root_cause.ipynb.
ROOT_CAUSE_TABLE = f"{CATALOG}.{ALYT}.deviation_root_cause"


class DeterministicContextParser:
    """Parses the deterministic_context blob into per-event entity-type arrays.

    The blob is a set of resolved-reference blocks joined by "---", each block
    starting with `[TYPE] "key"` and (usually) a `Canonical: <name>` line. Each
    block is routed to its typed bucket by TYPE_TO_BUCKET and de-duplicated in
    first-seen order.
    """

    # deterministic_context [TYPE] -> output entity bucket
    TYPE_TO_BUCKET = {
        "CLINICAL_ID": "clinical_ids",
        "DOCUMENT":    "documents",
        "ACRONYM":     "acronyms",
        "CRO":         "cros",
        "DEVICE":      "devices",
        "VENDOR":      "vendors",
    }

    def parse(self, ctx: str) -> dict:
        result = {bucket: [] for bucket in self.TYPE_TO_BUCKET.values()}
        if not ctx:
            return result

        for block in ctx.split("---"):
            block = block.strip()
            if not block:
                continue
            # First line of each block: [TYPE] "key"
            m = re.match(r'\[(\w+)\]\s*"([^"]+)"', block)
            if not m:
                continue
            bucket = self.TYPE_TO_BUCKET.get(m.group(1).upper())
            if bucket is None:
                continue
            # Prefer the resolved canonical name when present
            canon_match = re.search(r'Canonical:\s*(.+)', block)
            result[bucket].append(canon_match.group(1).strip() if canon_match else m.group(2))

        # De-duplicate while preserving order
        return {k: list(dict.fromkeys(v)) for k, v in result.items()}


# ---- Load per-event enrichment and parse ----------------------------------
embed_pdf = (
    spark.table(EMB_TABLE)
    .select("pr_id", "deterministic_context")
    .toPandas()
)
embed_pdf["pr_id"] = embed_pdf["pr_id"].astype(str)
embed_pdf["deterministic_context"] = embed_pdf["deterministic_context"].fillna("")

parser = DeterministicContextParser()
parsed = embed_pdf["deterministic_context"].apply(parser.parse)

entity_df = pd.DataFrame({
    "pr_id":                 embed_pdf["pr_id"],
    "clinical_ids":          parsed.apply(lambda x: x["clinical_ids"]),
    "documents":             parsed.apply(lambda x: x["documents"]),
    "acronyms":              parsed.apply(lambda x: x["acronyms"]),
    "cros":                  parsed.apply(lambda x: x["cros"]),
    "devices":               parsed.apply(lambda x: x["devices"]),
    "vendors":               parsed.apply(lambda x: x["vendors"]),
    "deterministic_context": embed_pdf["deterministic_context"],
})

print(f"Parsed {len(entity_df):,} events")
print(f"  with clinical_ids: {(entity_df['clinical_ids'].str.len() > 0).sum()}")
print(f"  with documents:    {(entity_df['documents'].str.len() > 0).sum()}")
print(f"  with acronyms:     {(entity_df['acronyms'].str.len() > 0).sum()}")
print(f"  with cros:         {(entity_df['cros'].str.len() > 0).sum()}")
print(f"  with devices:      {(entity_df['devices'].str.len() > 0).sum()}")
print(f"  with vendors:      {(entity_df['vendors'].str.len() > 0).sum()}")

In [0]:
# Cell 2 — LOC column: per-event country extraction
# ============================================================================
# LOC Column — per-event LOC flag and country extraction
# Mirrors build_reference_glossary dev_05b_extra logic.
# Output: loc_sdf (pr_id, LOC) — array<string> of inferred countries,
#         ['LOC'] if only structured flag, or empty array.
# ============================================================================
LOC_DICT_TABLE = f"{CATALOG}.{ALYT}.loc_country_dictionary"

# Load LOC country dictionary into temp view
ref_loc = (
    spark.table(LOC_DICT_TABLE)
         .selectExpr(
             "lower(trim(signal_type)) AS signal_type",
             "trim(pattern)            AS pattern",
             "trim(country)            AS country",
         )
         .filter("signal_type IS NOT NULL AND pattern IS NOT NULL AND country IS NOT NULL")
)
ref_loc.createOrReplaceTempView("ref_loc_country")
print(f"ref_loc_country loaded: {ref_loc.count()} rows")
ref_loc.groupBy("signal_type").count().orderBy("signal_type").show()

# Per-event: aggregate search text + operational_unit from source table
per_event = (
    spark.table(SOURCE_TABLE)
    .groupBy(F.col(f"`{SRC_ID_COL}`").cast("string").alias("pr_id"))
    .agg(
        F.concat_ws(" ",
            F.first("Event_Title", ignorenulls=True),
            F.first("Event_Description", ignorenulls=True),
        ).alias("event_search_text"),
        F.first("operational_unit", ignorenulls=True).alias("operational_unit"),
    )
)
per_event.createOrReplaceTempView("loc_events")

# LOC matching via SQL (adapted from build_reference_glossary dev_05b_extra)
loc_sdf = spark.sql(r"""
WITH sig AS (
    SELECT
        pr_id,
        event_search_text,
        operational_unit,
        coalesce(regexp_extract_all(event_search_text,
            '(?i)https?://[^\\s]+?\\.([a-z]{2})(?:[/:?#\\s)\\]]|$)', 1), array()) AS _cctld,
        flatten(transform(
            array_union(
                coalesce(regexp_extract_all(event_search_text, '\\b([A-Z]{2,3})\\s+LOC\\b', 1), array()),
                coalesce(regexp_extract_all(event_search_text, '\\bLOC\\s+([A-Z]{2,3}(?:/[A-Z]{2,3})*)\\b', 1), array())
            ),
            t -> split(t, '/')
        )) AS _adj
    FROM loc_events
),
m_text AS (
    SELECT s.pr_id, collect_set(r.country) AS countries
    FROM sig s
    JOIN ref_loc_country r
      ON ( r.signal_type = 'name'
           AND s.event_search_text rlike concat('(?i)\\b(', r.pattern, ')\\b') )
      OR ( r.signal_type = 'agency'
           AND s.event_search_text rlike concat('\\b(', r.pattern, ')\\b') )
    GROUP BY s.pr_id
),
m_tld AS (
    SELECT s.pr_id, collect_set(r.country) AS countries
    FROM (SELECT pr_id, explode(_cctld) AS tld FROM sig) s
    JOIN ref_loc_country r
      ON r.signal_type = 'cctld' AND lower(r.pattern) = lower(s.tld)
    GROUP BY s.pr_id
),
m_code AS (
    SELECT s.pr_id, collect_set(r.country) AS countries
    FROM (SELECT pr_id, explode(_adj) AS code FROM sig) s
    JOIN ref_loc_country r
      ON r.signal_type = 'code' AND upper(r.pattern) = upper(s.code)
    GROUP BY s.pr_id
),
joined AS (
    SELECT
        s.pr_id,
        s.operational_unit,
        array_distinct(array_compact(concat(
            coalesce(mt.countries, array()),
            coalesce(ml.countries, array()),
            coalesce(mc.countries, array())
        ))) AS loc_country_candidates
    FROM sig s
    LEFT JOIN m_text mt ON s.pr_id = mt.pr_id
    LEFT JOIN m_tld  ml ON s.pr_id = ml.pr_id
    LEFT JOIN m_code mc ON s.pr_id = mc.pr_id
)
SELECT
    pr_id,
    CASE
        WHEN size(loc_country_candidates) > 0
            THEN loc_country_candidates
        WHEN upper(trim(operational_unit)) = 'LOC'
            THEN array('LOC')
        ELSE array()
    END AS LOC
FROM joined
""")

n_loc = loc_sdf.filter(F.size("LOC") > 0).count()
print(f"LOC computed: {loc_sdf.count():,} events, {n_loc:,} LOC-tagged")

In [0]:
# Cell 3 — Assemble and write the supplementary table
# ============================================================================
# SUPPLEMENTARY TABLE  (STEP 2 of 2)
#
# Final columns (per event, one row per pr_id):
#   Event_Number              — event identifier
#   source_free_text_full     — clean deviation text (all free-text fields)
#   contextual_retrieval_text_full — enriched text (deterministic refs + LLM context + source)
#   deterministic_context     — raw resolved-references blob
#   clinical_ids              — array of parsed clinical IDs
#   documents                 — array of parsed document/SOP references
#   acronyms                  — array of parsed acronyms
#   cros                      — array of parsed CRO names
#   devices                   — array of parsed device names
#   vendors                   — array of parsed vendor names
#   personnel                 — array of people involved (Event_Owner, QA_Contact,
#                               Reported_by, Quality_Approver_CAPA_Final)
#   primary_topic             — main topic (zero-shot BERTopic BGE-M3)
#   secondary_topic           — second topic if multi-label (zero-shot BERT)
#   tertiary_topic            — third topic if multi-label (zero-shot BERT)
#   llm_rc_primary            — LLM-judged primary root-cause (Ishikawa 6M)
#   llm_rc_secondary          — first contributing factor from LLM
#   llm_rc_tertiary           — second contributing factor from LLM
# ============================================================================
# ---- per-event enrichment (pandas -> Spark, with explicit array schema) ----
enrich_schema = T.StructType([
    T.StructField("pr_id",                 T.StringType(),               False),
    T.StructField("clinical_ids",          T.ArrayType(T.StringType()),  True),
    T.StructField("documents",             T.ArrayType(T.StringType()),  True),
    T.StructField("acronyms",              T.ArrayType(T.StringType()),  True),
    T.StructField("cros",                  T.ArrayType(T.StringType()),  True),
    T.StructField("devices",               T.ArrayType(T.StringType()),  True),
    T.StructField("vendors",               T.ArrayType(T.StringType()),  True),
    T.StructField("deterministic_context", T.StringType(),               True),
])

enrich_rows = [
    (r.pr_id, list(r.clinical_ids), list(r.documents), list(r.acronyms),
     list(r.cros), list(r.devices), list(r.vendors),
     r.deterministic_context)
    for r in entity_df.itertuples(index=False)
]
enrich_sdf = spark.createDataFrame(enrich_rows, schema=enrich_schema)

# ---- Free-text columns from deviation_embeddings (fine-grained) ------------
text_sdf = (
    spark.table(EMB_TABLE)
    .select(
        F.col("pr_id").cast("string").alias("pr_id"),
        "source_free_text_full",
        "contextual_retrieval_text_full",
    )
)

# ---- Start with Event_Number as the key -----------------------------------
supp = (
    spark.table(SOURCE_TABLE)
    .select(F.col(f"`{SRC_ID_COL}`").cast("string").alias("pr_id"))
    .distinct()
)

# ---- Personnel involved (from source table name columns) -------------------
# Collect distinct non-null names from Event_Owner, QA_Contact, Reported_by,
# and Quality_Approver_CAPA_Final into a single array per event.
personnel_sdf = (
    spark.table(SOURCE_TABLE)
    .select(
        F.col(f"`{SRC_ID_COL}`").cast("string").alias("pr_id"),
        F.col("Event_Owner"),
        F.col("QA_Contact"),
        F.col("Reported_by"),
        F.col("Quality_Approver_CAPA_Final"),
    )
    .distinct()
    .withColumn(
        "personnel",
        F.array_distinct(
            F.filter(
                F.array(
                    F.col("Event_Owner"),
                    F.col("QA_Contact"),
                    F.col("Reported_by"),
                    F.col("Quality_Approver_CAPA_Final"),
                ),
                lambda x: x.isNotNull() & (F.trim(x) != F.lit(""))
            )
        )
    )
    .groupBy("pr_id")
    .agg(F.array_distinct(F.flatten(F.collect_list("personnel"))).alias("personnel"))
)
print(f"Personnel extracted for {personnel_sdf.count():,} events")

# ---- Join: text columns from embeddings table ----
supp = supp.join(text_sdf, on="pr_id", how="left")

# ---- Join: deterministic entities ----
supp = supp.join(enrich_sdf, on="pr_id", how="left")

# ---- Join: personnel ----
supp = supp.join(personnel_sdf, on="pr_id", how="left")

# ---- Join: LOC (country extraction from loc_country_dictionary) ----
supp = supp.join(loc_sdf, on="pr_id", how="left")

# ---- topic (zero-shot BERT: primary/secondary/tertiary from topic_modeling) --
if spark.catalog.tableExists(TOPIC_TABLE):
    topic_sdf = (
        spark.table(TOPIC_TABLE)
        .select(
            F.col("pr_id").cast("string").alias("pr_id"),
            F.col("primary_topic").cast("string"),
            F.col("secondary_topic").cast("string"),
            F.col("tertiary_topic").cast("string"),
        )
    )
    supp = supp.join(topic_sdf, on="pr_id", how="left")
    print(f"Joined topic assignments from {TOPIC_TABLE}")
else:
    supp = (
        supp
        .withColumn("primary_topic",   F.lit(None).cast("string"))
        .withColumn("secondary_topic", F.lit(None).cast("string"))
        .withColumn("tertiary_topic",  F.lit(None).cast("string"))
    )
    print(f"(!) {TOPIC_TABLE} not found — topic columns left null; "
          f"run topic_modeling notebook Cell 7 to populate them")

# ---- root-cause (LLM primary/secondary/tertiary from root_cause.ipynb) ------
if spark.catalog.tableExists(ROOT_CAUSE_TABLE):
    rc_sdf = (
        spark.table(ROOT_CAUSE_TABLE)
        .select(
            F.col("pr_id").cast("string").alias("pr_id"),
            F.col("root_cause_category").cast("string").alias("llm_rc_primary"),
            F.col("contributing_factors").getItem(0).cast("string").alias("llm_rc_secondary"),
            F.col("contributing_factors").getItem(1).cast("string").alias("llm_rc_tertiary"),
        )
    )
    supp = supp.join(rc_sdf, on="pr_id", how="left")
    print(f"Joined LLM root-cause columns from {ROOT_CAUSE_TABLE}")
else:
    supp = (
        supp
        .withColumn("llm_rc_primary",    F.lit(None).cast("string"))
        .withColumn("llm_rc_secondary",  F.lit(None).cast("string"))
        .withColumn("llm_rc_tertiary",   F.lit(None).cast("string"))
    )
    print(f"(!) {ROOT_CAUSE_TABLE} not found — root-cause columns left null; "
          f"run root_cause.ipynb to populate them")

# ---- Rename pr_id back to Event_Number and select final columns ------------
supp = supp.withColumnRenamed("pr_id", SRC_ID_COL)

FINAL_COLUMNS = [
    SRC_ID_COL,
    "source_free_text_full",
    "contextual_retrieval_text_full",
    "deterministic_context",
    "clinical_ids",
    "documents",
    "acronyms",
    "cros",
    "devices",
    "vendors",
    "personnel",
    "LOC",
    "primary_topic",
    "secondary_topic",
    "tertiary_topic",
    "llm_rc_primary",
    "llm_rc_secondary",
    "llm_rc_tertiary",
]
supp = supp.select(*FINAL_COLUMNS)

# ---- Write the supplementary table ----------------------------------------
supp.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)

n_rows = spark.table(OUTPUT_TABLE).count()
n_events = spark.table(OUTPUT_TABLE).select(f"`{SRC_ID_COL}`").distinct().count()
print(f"\n{OUTPUT_TABLE}")
print(f"  {n_rows:,} rows across {n_events:,} distinct events (one row per event)")

display(spark.table(OUTPUT_TABLE).limit(10))
